In [1]:
import os
# os.environ["TRANSFORMERS_NO_ACCELERATE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1" #离线加载

In [2]:
import jieba
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from torch.distributed.pipelining import pipeline
from umap import UMAP
import numpy as np
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

C:\Anaconda\envs\py39\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import torch
print(torch.cuda.is_available())
# torch.set_default_device('cuda')
torch.__version__

True


'2.8.0+cu126'

In [4]:
# 导入数据
# with open('陕西-3.txt', 'r', encoding='utf-8') as f:
#     docs=f.readlines()
with open('党史数据库切词.txt', 'r', encoding='utf-8') as f:
    docs=f.readlines()
print("文本数：",len(docs))
print("第一条：",docs[0])
vectorizer_model = None

文本数： 2410
第一条： 二零一二年



In [5]:
# set HF_ENDPOINT=https://hf-mirror.com
# 0 sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2\\snapshots\\86741b4e3f5cb7765a600d3a3d55a0f6a6cb443d")
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--KaLM-Embedding--KaLM-embedding-multilingual-mini-instruct-v2.5\\snapshots\\753c6fe26abc20a32aeb162003aa03457d15db2f")
from transformers import pipeline
from sentence_transformers import SentenceTransformer
# print(transformers.__version__)

from transformers import AutoModel, AutoTokenizer

# model_name = "bert-base-chinese"
# model = AutoModel.from_pretrained(model_name).to("cuda")
# tokenizer = AutoTokenizer.from_pretrained(model_name)
#
# embedding_model = pipeline("feature-extraction", model=model, tokenizer=tokenizer,device=0)

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('bert-base-chinese', device='cuda')
embeddings=np.load('emb_党史数据库.npy')
# print(1)

No sentence-transformers model found with name bert-base-chinese. Creating a new one with mean pooling.


In [6]:
umap_model=UMAP(
    n_neighbors=5,  #参考周围的5个连续点
    n_components=5, #将高维数据降维到5维
    min_dist=0.0,   #嵌入点之间的最小的有效距离，值越小会导致越密集的聚类（会减少离群值），默认为0.1
    metric='cosine',    #余弦距离，默认会使用欧几里得距离
    random_state=95 #设置随机数种子
)

In [7]:
hdbscan_model=HDBSCAN(
    min_cluster_size=5,
    min_samples=5,  #减少该值可以减少离群值，默认等于min_cluster_size
    metric='euclidean',
    # prediction_data=True,   #若计算文本属于每一个主题的概率，则必须为True
)   #每个类中至少包含5条

In [8]:
# def tokenize_zh(text):  #切词
#     words=jieba.cut(text)
#     return words
#
# # 若导入的是切词后的文件，则下一条语句无需填写参数
# vectorizer_model=CountVectorizer(tokenizer=tokenize_zh)
vectorizer_model=CountVectorizer(stop_words=['二零一二','二零一三','二零一四','二零一五','二零一六','二零一七','二零一八','二零一九','二零二零','二零二一','二零二二','二零二三','二零二四','二零二五','二零二六'])

if torch.cuda.is_available():
    print(f"当前可用 GPU: {torch.cuda.device_count()}")
    print(f"GPU 0 名称: {torch.cuda.get_device_name(0)}")

当前可用 GPU: 1
GPU 0 名称: NVIDIA GeForce RTX 4060 Laptop GPU


In [9]:
print("embedding_model 设备:", embedding_model.device)

embedding_model 设备: cuda:0


In [10]:
topic_model=BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    # min_topic_size=5, #每个主题至少包含5条，有min_cluster_size就不用设置
    # nr_topics='auto',   #自动合并主题
    # calculate_probabilities=True,   #计算属于每一个主题的概率
)
# print(embeddings.shape)
# print(len(docs))
topics, probs=topic_model.fit_transform(docs)
topic_info=topic_model.get_topic_info()
topic_info
topic_info.to_csv('topic_info.csv', index=False, encoding='utf-8-sig')

In [11]:
print(len(topics),topics[:10])
print(len(probs),probs[:10])

2410 [38, 23, -1, -1, 54, -1, 45, -1, -1, 96]
2410 [1.         1.         0.         0.         0.89858795 0.
 0.63677196 0.         0.         1.        ]


In [12]:
# topic_model.set_topic_labels({
#     0:'',
#     1:'',
#     2:''
# })
# topic_info=topic_model.get_topic_info()
# topic_info
# topic_model.visualize_barchart(custom_labels=True)
topic_model.visualize_barchart()

In [13]:
topic_model.visualize_topics()

In [14]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, hide_document_hover=True)

In [15]:
import plotly.express as px
hierarchical_topics = topic_model.hierarchical_topics(docs)
# topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

# 为每个轨迹随机分配颜色（或使用预定义颜色列表）
colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']
for i, trace in enumerate(fig.data):
    trace.line.color = colors[i % len(colors)]

fig.show()
fig.write_html('topic_hierarchy.html')

100%|██████████| 112/112 [00:00<00:00, 146.28it/s]


In [16]:
with open('党史数据库时间.txt','r',encoding='utf-8') as file:
    lines=file.readlines()
    timestamps=[int(line.strip()) for line in lines]
print(len(timestamps),timestamps[:10])

2410 [2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012, 2012]


In [17]:
topics_over_time=topic_model.topics_over_time(docs,timestamps,global_tuning=False, evolution_tuning=False) # global_tuning=False, evolution_tuning=False
topic_model.visualize_topics_over_time(topics_over_time)

In [18]:
topic_docs = topic_model.get_document_info(docs)
topic_docs.to_csv('党史数据库聚类结果.csv')